# Multi-Head Attention — Exercise

Companion to [Attention Part 1 § Multi-Head Attention](https://tanulsingh.github.io/blog/attention-mechanisms#multi-head-attention-why-one-perspective-isnt-enough).

You'll build multi-head attention using the **reshape-then-split** pattern (View B from the blog) — this is what real PyTorch/HuggingFace implementations do, because it's far more GPU-efficient than running `h` separate small attentions.

**The key trick:** one big `(d_model, d_model)` weight matrix → reshape into `(n_heads, d_head)` slices.

## Setup

In [ ]:
# TODO: imports — torch, torch.nn as nn, torch.nn.functional as F, math

## TODO 1 — `MultiHeadAttention` module

Build the module step by step. Inputs are shape `(batch, seq_len, d_model)`. The plan:

1. **Project** `x` to Q, K, V via three big linear layers, each `nn.Linear(d_model, d_model)`. Each output is shape `(batch, seq_len, d_model)`.
2. **Reshape** each to `(batch, seq_len, n_heads, d_head)` where `d_head = d_model // n_heads`.
3. **Transpose** to `(batch, n_heads, seq_len, d_head)` so the attention computation can be batched over the `n_heads` axis.
4. **Compute attention** using your `scaled_dot_product_attention` from earlier exercises (with optional causal mask).
5. **Transpose back** to `(batch, seq_len, n_heads, d_head)` and reshape to `(batch, seq_len, d_model)`.
6. **Output projection** through `W_O` (`nn.Linear(d_model, d_model)`).

The output projection at the end is what mixes information from different heads.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, causal: bool = False):
        super().__init__()
        # TODO 1a: assert d_model % n_heads == 0
        # TODO 1b: store d_model, n_heads, d_head = d_model // n_heads, causal
        # TODO 1c: create W_q, W_k, W_v, W_o as nn.Linear(d_model, d_model)
        pass

    def forward(self, x):
        # x: (batch, seq_len, d_model)
        batch, seq_len, _ = x.shape

        # TODO 1d: project to Q, K, V — each (batch, seq_len, d_model)

        # TODO 1e: reshape and transpose to (batch, n_heads, seq_len, d_head)
        #   tensor.view(batch, seq_len, n_heads, d_head).transpose(1, 2)

        # TODO 1f: build causal mask if self.causal, else mask=None

        # TODO 1g: run scaled_dot_product_attention(Q, K, V, mask)
        #   Output shape: (batch, n_heads, seq_len, d_head)

        # TODO 1h: transpose back and reshape to (batch, seq_len, d_model)
        #   .transpose(1, 2).contiguous().view(batch, seq_len, d_model)

        # TODO 1i: apply W_o

        pass

In [ ]:
# Sanity check:
#   - mha = MultiHeadAttention(d_model=64, n_heads=8)
#   - output = mha(torch.randn(2, 10, 64))
#   - output.shape == (2, 10, 64)
#   - n_params = sum(p.numel() for p in mha.parameters())
#     should be 4 * (64*64 + 64) for [W_q, W_k, W_v, W_o] linear layers

## TODO 2 — Verify causal MultiHeadAttention

Instantiate with `causal=True`, run on a sequence, and verify (using the same trick as exercise 02) that changing `V[t]` for `t > i` does not affect `output[i]`.

In [ ]:
# TODO 2:
#   - mha = MultiHeadAttention(d_model=64, n_heads=8, causal=True)
#   - x_a = torch.randn(1, 5, 64)
#   - x_b = x_a.clone()
#   - x_b[0, 4] = torch.randn(64)  # change last token only
#   - out_a = mha(x_a); out_b = mha(x_b)
#   - Assert torch.allclose(out_a[0, :4], out_b[0, :4], atol=1e-5)  # earlier positions same
#   - Assert not torch.allclose(out_a[0, 4], out_b[0, 4])           # last position different

## Run the tests

In [ ]:
from tests import run_multi_head_tests
# run_multi_head_tests(MultiHeadAttention)